In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("module_11_assignment").getOrCreate()

PySpark Assignment Questions
syntax
1. Basic DataFrame Operations
 Load the sales.csv and customer.csv files into separate DataFrames.

In [2]:
df_customer = spark.read.csv("/content/customers_dirty.csv", header=True, inferSchema=True)
df_sales = spark.read.csv("/content/sales_dirty.csv", header=True, inferSchema=True)

In [3]:
df_customer.show()
df_sales.show()


+-----------+------------------+--------------------+---+-----------------+
|customer_id|     customer_name|               email|age|             city|
+-----------+------------------+--------------------+---+-----------------+
|        100| Jasmine Contreras| xmacias@example.org| 49|      Hollandtown|
|        101|     Michael Jones|michaelkhan@examp...| 27|  New Kennethstad|
|        102|  Dr. Jason Murray|penapatricia@exam...| 37|       West Megan|
|        103|   Karen Hernandez|                NULL| 28|   South Courtney|
|        104|  Kathleen Chapman|jessejones@exampl...| 45|      South Paige|
|        105|    Melissa Greene|deanaustin@exampl...| 24|      Charlesberg|
|        106|       Brenda Ruiz|                NULL| 32|        Lake Erin|
|        107| Dr. Paul Bautista|zbrennan@example.com| 37|      North Kayla|
|        108|      Samuel Garza|edward54@example.net| 33|     East Michael|
|        109|       Lori Morris|pricesusan@exampl...| 46|    New Jamesview|
|        110

2. Display the schema of both DataFrames.

In [4]:
df_sales.printSchema()
df_customer.printSchema()

root
 |-- sales_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- region: string (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)



3. Show the first 5 rows from the sales DataFrame.

In [5]:
df_sales.show(5)

+--------+-----------+-------+-------+----------+------+
|sales_id|customer_id|product| amount| sale_date|region|
+--------+-----------+-------+-------+----------+------+
|       1|       4296| Laptop|10299.0|2025-04-08| South|
|       2|       6307| Laptop|16842.0|2025-11-25|  East|
|       3|       5369|Desktop|64411.0|2025-04-13|  East|
|       4|       6125| Mobile|55543.0|2024-12-15| North|
|       5|       9800| Laptop|48545.0|2024-05-11|  East|
+--------+-----------+-------+-------+----------+------+
only showing top 5 rows


4. Count the number of rows and columns in the customer DataFrame.

In [6]:
print('rows')
print(df_customer.count())
print('columns')
print(len(df_customer.columns))

rows
11000
columns
5


 Data Cleaning
5. Remove duplicate rows from the sales DataFrame based on
customer_id,product,amount,sale_date,region columns

In [7]:
print(df_sales.count())

110000


In [8]:
df_dropped = df_sales.dropDuplicates(['customer_id', 'product', 'amount', 'sale_date', 'region'])
print(df_dropped.count())

100000


6. Drop rows where any column in the customer DataFrame has null values.

In [9]:
df_mod = df_customer.na.drop()
df_mod.count()

9940

7. Replace null values in the amount column of the sales DataFrame with 0.

In [10]:
df_sales = df_sales.na.fill(0, 'amount')
df_sales.show()

+--------+-----------+-------+-------+----------+------+
|sales_id|customer_id|product| amount| sale_date|region|
+--------+-----------+-------+-------+----------+------+
|       1|       4296| Laptop|10299.0|2025-04-08| South|
|       2|       6307| Laptop|16842.0|2025-11-25|  East|
|       3|       5369|Desktop|64411.0|2025-04-13|  East|
|       4|       6125| Mobile|55543.0|2024-12-15| North|
|       5|       9800| Laptop|48545.0|2024-05-11|  East|
|       6|       5535| Laptop|73010.0|2025-10-06| North|
|       7|       8351| Laptop|15313.0|2024-07-13| North|
|       8|        247| Laptop|50976.0|2025-05-20| South|
|       9|       4285| Laptop|98067.0|2024-06-01|  West|
|      10|       2987| Laptop|11911.0|2025-04-08| North|
|      11|       2032|Desktop|79916.0|2025-05-05|  West|
|      12|       8810| Tablet|32820.0|2024-08-25| South|
|      13|       8932| Laptop|57096.0|2025-09-23| South|
|      14|       7953| Tablet|60109.0|2024-05-31|  West|
|      15|       5838|Desktop|3

8. Replace null values in the email column of the customer DataFrame with the value
"unknown".

In [11]:
df_customer = df_customer.na.fill('unknown', 'email')
df_customer.show()

+-----------+------------------+--------------------+---+-----------------+
|customer_id|     customer_name|               email|age|             city|
+-----------+------------------+--------------------+---+-----------------+
|        100| Jasmine Contreras| xmacias@example.org| 49|      Hollandtown|
|        101|     Michael Jones|michaelkhan@examp...| 27|  New Kennethstad|
|        102|  Dr. Jason Murray|penapatricia@exam...| 37|       West Megan|
|        103|   Karen Hernandez|             unknown| 28|   South Courtney|
|        104|  Kathleen Chapman|jessejones@exampl...| 45|      South Paige|
|        105|    Melissa Greene|deanaustin@exampl...| 24|      Charlesberg|
|        106|       Brenda Ruiz|             unknown| 32|        Lake Erin|
|        107| Dr. Paul Bautista|zbrennan@example.com| 37|      North Kayla|
|        108|      Samuel Garza|edward54@example.net| 33|     East Michael|
|        109|       Lori Morris|pricesusan@exampl...| 46|    New Jamesview|
|        110

9. Add a new column discounted_amount to the sales DataFrame that applies a 10%
discount on amount.

In [12]:
df_sales = df_sales.withColumn('discounted_amount', col('amount') * 0.9)
df_sales.show()

+--------+-----------+-------+-------+----------+------+------------------+
|sales_id|customer_id|product| amount| sale_date|region| discounted_amount|
+--------+-----------+-------+-------+----------+------+------------------+
|       1|       4296| Laptop|10299.0|2025-04-08| South|            9269.1|
|       2|       6307| Laptop|16842.0|2025-11-25|  East|15157.800000000001|
|       3|       5369|Desktop|64411.0|2025-04-13|  East|           57969.9|
|       4|       6125| Mobile|55543.0|2024-12-15| North|49988.700000000004|
|       5|       9800| Laptop|48545.0|2024-05-11|  East|           43690.5|
|       6|       5535| Laptop|73010.0|2025-10-06| North|           65709.0|
|       7|       8351| Laptop|15313.0|2024-07-13| North|           13781.7|
|       8|        247| Laptop|50976.0|2025-05-20| South|           45878.4|
|       9|       4285| Laptop|98067.0|2024-06-01|  West|           88260.3|
|      10|       2987| Laptop|11911.0|2025-04-08| North|           10719.9|
|      11|  

10. Rename the city column in the customer DataFrame to customer_city.

In [13]:
df_customer = df_customer.withColumnRenamed('city', 'customer_city')
df_customer.show()

+-----------+------------------+--------------------+---+-----------------+
|customer_id|     customer_name|               email|age|    customer_city|
+-----------+------------------+--------------------+---+-----------------+
|        100| Jasmine Contreras| xmacias@example.org| 49|      Hollandtown|
|        101|     Michael Jones|michaelkhan@examp...| 27|  New Kennethstad|
|        102|  Dr. Jason Murray|penapatricia@exam...| 37|       West Megan|
|        103|   Karen Hernandez|             unknown| 28|   South Courtney|
|        104|  Kathleen Chapman|jessejones@exampl...| 45|      South Paige|
|        105|    Melissa Greene|deanaustin@exampl...| 24|      Charlesberg|
|        106|       Brenda Ruiz|             unknown| 32|        Lake Erin|
|        107| Dr. Paul Bautista|zbrennan@example.com| 37|      North Kayla|
|        108|      Samuel Garza|edward54@example.net| 33|     East Michael|
|        109|       Lori Morris|pricesusan@exampl...| 46|    New Jamesview|
|        110

11. Drop the region column from the sales DataFrame.

In [14]:
df_dropped = df_dropped.drop('region')
df_dropped.show()

+--------+-----------+-------+-------+----------+
|sales_id|customer_id|product| amount| sale_date|
+--------+-----------+-------+-------+----------+
|      65|       1572| Laptop|74830.0|2025-04-04|
|     147|       5956|Desktop|97967.0|2025-05-11|
|     536|       9591|Desktop|59854.0|2025-02-12|
|     709|       1464| Laptop|14835.0|2025-10-09|
|    1121|       9844| Mobile|99086.0|2024-12-02|
|    1625|       6207| Laptop|42745.0|2025-02-22|
|    1637|       4672| Laptop|64739.0|2024-06-18|
|    1678|       2480| Tablet|52089.0|2024-06-27|
|    1997|       2690| Laptop|73162.0|2025-10-28|
|    3029|       7997| Laptop|73808.0|2024-07-19|
|    3381|       1620| Laptop|95140.0|2025-09-19|
|    3386|       5732| Laptop|66298.0|2025-08-12|
|    3508|       2495| Tablet|94119.0|2025-09-18|
|    3800|       1942| Mobile|16922.0|2025-10-05|
|    4205|       3528| Laptop|38636.0|2024-06-14|
|    4371|       2460| Laptop|81481.0|2024-06-03|
|    4583|       7938|Desktop|14870.0|2026-02-19|


12. Create a new column customer_age_category in the customer DataFrame based on age:
a. "Youth" for age < 30
b. "Adult" for 30 <= age < 50
c. "Senior" for age >= 50

In [15]:
df_customer = df_customer.withColumn('customer_age_category', when(col('age') < 30, 'Young').when((col('age') >= 30) & (col('age') < 50),'Adult').when(col('age') >= 50, 'Senior'))
df_customer.show()

+-----------+------------------+--------------------+---+-----------------+---------------------+
|customer_id|     customer_name|               email|age|    customer_city|customer_age_category|
+-----------+------------------+--------------------+---+-----------------+---------------------+
|        100| Jasmine Contreras| xmacias@example.org| 49|      Hollandtown|                Adult|
|        101|     Michael Jones|michaelkhan@examp...| 27|  New Kennethstad|                Young|
|        102|  Dr. Jason Murray|penapatricia@exam...| 37|       West Megan|                Adult|
|        103|   Karen Hernandez|             unknown| 28|   South Courtney|                Young|
|        104|  Kathleen Chapman|jessejones@exampl...| 45|      South Paige|                Adult|
|        105|    Melissa Greene|deanaustin@exampl...| 24|      Charlesberg|                Young|
|        106|       Brenda Ruiz|             unknown| 32|        Lake Erin|                Adult|
|        107| Dr. Pa

13. Filter the sales DataFrame to show only rows where amount is greater than 50,000.

In [16]:
stat1 = df_sales.filter(df_sales.amount > 50000)
stat1.show()
stat1.count()

+--------+-----------+-------+-------+----------+------+------------------+
|sales_id|customer_id|product| amount| sale_date|region| discounted_amount|
+--------+-----------+-------+-------+----------+------+------------------+
|       3|       5369|Desktop|64411.0|2025-04-13|  East|           57969.9|
|       4|       6125| Mobile|55543.0|2024-12-15| North|49988.700000000004|
|       6|       5535| Laptop|73010.0|2025-10-06| North|           65709.0|
|       8|        247| Laptop|50976.0|2025-05-20| South|           45878.4|
|       9|       4285| Laptop|98067.0|2024-06-01|  West|           88260.3|
|      11|       2032|Desktop|79916.0|2025-05-05|  West| 71924.40000000001|
|      13|       8932| Laptop|57096.0|2025-09-23| South|           51386.4|
|      14|       7953| Tablet|60109.0|2024-05-31|  West|           54098.1|
|      16|       1936| Tablet|69796.0|2026-03-25| South|           62816.4|
|      20|       1409| Tablet|58564.0|2026-01-23| South|           52707.6|
|      22|  

54145

In [53]:
df_sales.createOrReplaceTempView("sales_table")
stat1_sql = spark.sql("SELECT * FROM sales_table WHERE amount > 50000")
stat1_sql.show()
stat1_sql.count()

+--------+-----------+-------+-------+----------+------+-----------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|
+--------+-----------+-------+-------+----------+------+-----------------+
|    3486|       1646| Laptop|99999.0|2024-12-13| South|          89999.1|
|   69049|       6637| Laptop|99999.0|2025-05-10| South|          89999.1|
|    6874|       6345| Tablet|99997.0|2025-06-02| South|          89997.3|
|   96541|       9079| Laptop|99995.0|2024-05-26| North|          89995.5|
|   37866|        636| Mobile|99995.0|2025-05-07|  West|          89995.5|
|   96541|       9079| Laptop|99995.0|2024-05-26| North|          89995.5|
|    8104|       6810| Laptop|99993.0|2024-06-28|  West|          89993.7|
|   71825|       5731| Mobile|99993.0|2024-06-15| South|          89993.7|
|   78912|       9982| Laptop|99993.0|2025-12-31|  West|          89993.7|
|   20093|       6368|Desktop|99992.0|2025-06-13|  East|          89992.8|
|   70208|       8097| La

54145

14. Filter the customer DataFrame to show customers aged between 25 and 30.

In [17]:
stat2 = df_customer.filter((df_customer.age >= 25) & (df_customer.age <= 30))
stat2.show()
stat2.count()

+-----------+-----------------+--------------------+---+------------------+---------------------+
|customer_id|    customer_name|               email|age|     customer_city|customer_age_category|
+-----------+-----------------+--------------------+---+------------------+---------------------+
|        101|    Michael Jones|michaelkhan@examp...| 27|   New Kennethstad|                Young|
|        103|  Karen Hernandez|             unknown| 28|    South Courtney|                Young|
|        138|    Chelsea Ortiz|kellycollins@exam...| 29|         Emilyview|                Young|
|        140|      Eric Cortez|elaine06@example.org| 29|         Meganfurt|                Young|
|        153|   Timothy Miller|smithjack@example...| 26|        Conniestad|                Young|
|        162|Justin Coleman MD|  john68@example.com| 28|        Pamelastad|                Young|
|        171|  Brooke Gonzales| james62@example.net| 29|        Lake Jacob|                Young|
|        202|       

1225

In [52]:
df_customer.createOrReplaceTempView('customer_table')
stat2_sql = spark.sql("SELECT * FROM customer_table WHERE age >= 25 AND age <= 30")
stat2_sql.show()
stat2_sql.count()

+-----------+-----------------+--------------------+---+--------------------+---------------------+
|customer_id|    customer_name|               email|age|       customer_city|customer_age_category|
+-----------+-----------------+--------------------+---+--------------------+---------------------+
|        522|     Cameron Diaz| orhodes@example.net| 25|       Charlotteport|                Young|
|        801|     Linda Robles|             unknown| 25|      West Scottbury|                Young|
|        653|      Justin Bell|  operez@example.net| 25|South Brittanyche...|                Young|
|        295|Melissa Olson PhD|  bmoran@example.net| 25|              Kimton|                Young|
|        665|     Michael Pham|  xhines@example.org| 25|     Mitchellchester|                Young|
|        340|    Matthew Smith|  john70@example.org| 25|  South Charlesshire|                Young|
|        675|Danielle Trujillo|wheelerjeffrey@ex...| 25|         Vanessaland|                Young|


1225

15. Identify all customers who have made purchases in more than one region.

In [18]:
stat3 = df_sales.groupBy('customer_id').agg(F.countDistinct('region').alias('region_count'))
stat3.show()
stat3 = stat3.filter(stat3.region_count > 1)
stat3.show()
stat3.count()

+-----------+------------+
|customer_id|region_count|
+-----------+------------+
|        463|           4|
|       2142|           3|
|       3175|           3|
|       3918|           4|
|       7253|           4|
|       1238|           4|
|       6357|           4|
|       8638|           4|
|       6336|           3|
|       3794|           4|
|       5518|           4|
|       4900|           3|
|       9900|           4|
|        833|           4|
|       5156|           3|
|       1342|           4|
|       1829|           3|
|       1645|           4|
|       1088|           4|
|       6466|           4|
+-----------+------------+
only showing top 20 rows
+-----------+------------+
|customer_id|region_count|
+-----------+------------+
|        463|           4|
|       2142|           3|
|       3175|           3|
|       3918|           4|
|       7253|           4|
|       1238|           4|
|       6357|           4|
|       8638|           4|
|       6336|           3|
|  

9982

16. Filter the top 3 sales based on amount for each product.

In [21]:
stat4 = df_sales.withColumn('rank', rank().over(Window.partitionBy('product').orderBy(desc('amount'))))
stat4 = stat4.filter(stat4.rank <= 3)
stat4.show()

+--------+-----------+-------+-------+----------+------+-----------------+----+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|rank|
+--------+-----------+-------+-------+----------+------+-----------------+----+
|   20093|       6368|Desktop|99992.0|2025-06-13|  East|          89992.8|   1|
|   28608|       3517|Desktop|99985.0|2024-05-19| North|          89986.5|   2|
|    3913|       2539|Desktop|99968.0|2025-03-04| South|          89971.2|   3|
|    3486|       1646| Laptop|99999.0|2024-12-13| South|          89999.1|   1|
|   69049|       6637| Laptop|99999.0|2025-05-10| South|          89999.1|   1|
|   96541|       9079| Laptop|99995.0|2024-05-26| North|          89995.5|   3|
|   96541|       9079| Laptop|99995.0|2024-05-26| North|          89995.5|   3|
|   37866|        636| Mobile|99995.0|2025-05-07|  West|          89995.5|   1|
|   71825|       5731| Mobile|99993.0|2024-06-15| South|          89993.7|   2|
|   42165|       2101| Mobile|99982.0|20

17. Perform an inner join between sales and customer DataFrames on customer_id.

In [22]:
joined1 = df_sales.join(df_customer,'customer_id', 'inner')
joined1.show()

+-----------+--------+-------+-------+----------+------+------------------+--------------------+--------------------+---+----------------+---------------------+
|customer_id|sales_id|product| amount| sale_date|region| discounted_amount|       customer_name|               email|age|   customer_city|customer_age_category|
+-----------+--------+-------+-------+----------+------+------------------+--------------------+--------------------+---+----------------+---------------------+
|       4296|       1| Laptop|10299.0|2025-04-08| South|            9269.1|       Jeffrey Boone| jacob58@example.com| 63|West Christopher|               Senior|
|       6307|       2| Laptop|16842.0|2025-11-25|  East|15157.800000000001|      Christopher Ho| laura12@example.org| 33|       Patelfurt|                Adult|
|       5369|       3|Desktop|64411.0|2025-04-13|  East|           57969.9|       Ronald Arnold|alvaradomelissa@e...| 69|    South Thomas|               Senior|
|       5369|       3|Desktop|6441

In [55]:
df_sales.createOrReplaceTempView('sales_table')
df_customer.createOrReplaceTempView('customer_table')
joined2_sql = spark.sql("SELECT s.*, c.customer_name, c.email, c.age, c.customer_city, c.customer_age_category FROM sales_table s INNER JOIN customer_table c ON s.customer_id = c.customer_id")
joined2_sql.show()

+--------+-----------+-------+-------+----------+------+------------------+--------------------+--------------------+---+----------------+---------------------+
|sales_id|customer_id|product| amount| sale_date|region| discounted_amount|       customer_name|               email|age|   customer_city|customer_age_category|
+--------+-----------+-------+-------+----------+------+------------------+--------------------+--------------------+---+----------------+---------------------+
|       1|       4296| Laptop|10299.0|2025-04-08| South|            9269.1|       Jeffrey Boone| jacob58@example.com| 63|West Christopher|               Senior|
|       2|       6307| Laptop|16842.0|2025-11-25|  East|15157.800000000001|      Christopher Ho| laura12@example.org| 33|       Patelfurt|                Adult|
|       3|       5369|Desktop|64411.0|2025-04-13|  East|           57969.9|       Ronald Arnold|alvaradomelissa@e...| 69|    South Thomas|               Senior|
|       3|       5369|Desktop|6441

18. Perform a left join to include all records from sales and matching records from
customer.

In [23]:
joined2 = df_sales.join(df_customer,'customer_id', 'left')
joined2.show()

+-----------+--------+-------+-------+----------+------+------------------+--------------------+--------------------+---+----------------+---------------------+
|customer_id|sales_id|product| amount| sale_date|region| discounted_amount|       customer_name|               email|age|   customer_city|customer_age_category|
+-----------+--------+-------+-------+----------+------+------------------+--------------------+--------------------+---+----------------+---------------------+
|       4296|       1| Laptop|10299.0|2025-04-08| South|            9269.1|       Jeffrey Boone| jacob58@example.com| 63|West Christopher|               Senior|
|       6307|       2| Laptop|16842.0|2025-11-25|  East|15157.800000000001|      Christopher Ho| laura12@example.org| 33|       Patelfurt|                Adult|
|       5369|       3|Desktop|64411.0|2025-04-13|  East|           57969.9|       Ronald Arnold|alvaradomelissa@e...| 69|    South Thomas|               Senior|
|       5369|       3|Desktop|6441

In [54]:
df_sales.createOrReplaceTempView('sales_table')
df_customer.createOrReplaceTempView('customer_table')
joined2_sql = spark.sql("SELECT s.*, c.customer_name, c.email, c.age, c.customer_city, c.customer_age_category FROM sales_table s LEFT JOIN customer_table c ON s.customer_id = c.customer_id")
joined2_sql.show()

+--------+-----------+-------+-------+----------+------+------------------+--------------------+--------------------+---+----------------+---------------------+
|sales_id|customer_id|product| amount| sale_date|region| discounted_amount|       customer_name|               email|age|   customer_city|customer_age_category|
+--------+-----------+-------+-------+----------+------+------------------+--------------------+--------------------+---+----------------+---------------------+
|       1|       4296| Laptop|10299.0|2025-04-08| South|            9269.1|       Jeffrey Boone| jacob58@example.com| 63|West Christopher|               Senior|
|       2|       6307| Laptop|16842.0|2025-11-25|  East|15157.800000000001|      Christopher Ho| laura12@example.org| 33|       Patelfurt|                Adult|
|       3|       5369|Desktop|64411.0|2025-04-13|  East|           57969.9|       Ronald Arnold|alvaradomelissa@e...| 69|    South Thomas|               Senior|
|       3|       5369|Desktop|6441

19. Perform a full outer join between sales and customer DataFrames

In [24]:
joined3 = df_sales.join(df_customer,'customer_id', 'full')
joined3.show()

+-----------+--------+-------+-------+----------+------+------------------+---------------+--------------------+---+---------------+---------------------+
|customer_id|sales_id|product| amount| sale_date|region| discounted_amount|  customer_name|               email|age|  customer_city|customer_age_category|
+-----------+--------+-------+-------+----------+------+------------------+---------------+--------------------+---+---------------+---------------------+
|        101|    4026| Laptop|31135.0|2025-08-01| North|           28021.5|  Michael Jones|michaelkhan@examp...| 27|New Kennethstad|                Young|
|        101|   11711| Mobile|96542.0|2024-12-18|  West|           86887.8|  Michael Jones|michaelkhan@examp...| 27|New Kennethstad|                Young|
|        101|   44312| Laptop|64994.0|2025-03-17| North|           58494.6|  Michael Jones|michaelkhan@examp...| 27|New Kennethstad|                Young|
|        101|   49094| Mobile| -100.0|2025-02-22| South|             -

20. Identify customers who have not made any purchases by performing an anti-join.

In [25]:
joined4 = df_customer.join(df_sales, 'customer_id', 'anti')
joined4.show()

+-----------+-------------+-----+---+-------------+---------------------+
|customer_id|customer_name|email|age|customer_city|customer_age_category|
+-----------+-------------+-----+---+-------------+---------------------+
+-----------+-------------+-----+---+-------------+---------------------+



21. Calculate the total sales amount for each product.

In [26]:
stat5 = df_sales.groupBy('product').agg(sum('amount'))
stat5.show()

+-------+-------------+
|product|  sum(amount)|
+-------+-------------+
| Laptop|3.370429988E9|
| Mobile| 6.67684833E8|
| Tablet| 6.82077488E8|
|Desktop| 6.66552304E8|
+-------+-------------+



22. Find the average age of customers in the customer DataFrame.

In [27]:
stat6 = df_customer.agg(avg('age'))
stat6.show()

+------------------+
|          avg(age)|
+------------------+
|43.783727272727276|
+------------------+



23. Calculate the maximum and minimum sales amounts in the sales DataFrame.

In [28]:
stat7 = df_sales.agg(max('amount'), min('amount'))
stat7.show()

+-----------+-----------+
|max(amount)|min(amount)|
+-----------+-----------+
|    99999.0|     -100.0|
+-----------+-----------+



24. Group the customer DataFrame by customer_city and count the number of customers in
each city.

In [29]:
stat8 = df_customer.groupBy('customer_city').count()
stat8.show()

+------------------+-----+
|     customer_city|count|
+------------------+-----+
|         Dianaland|    1|
|     East Amymouth|    2|
|  South Nicoleport|    1|
|        Denisefurt|    1|
|   New Michelefurt|    1|
|       Port Monica|    1|
|          Annefurt|    1|
|       Lake Joshua|    7|
|     Wilsonchester|    2|
|     New Scottstad|    1|
|   South Amberfurt|    2|
| New Jennifershire|    2|
|         Austinton|    3|
|        East Tammy|    4|
|South Nicholastown|    1|
|      East Jeffrey|    2|
|          Karaland|    1|
|         East Cory|    1|
|        Karenmouth|    1|
|        Lake April|    2|
+------------------+-----+
only showing top 20 rows


25. Sort the sales DataFrame by amount in descending order.

In [30]:
df_sales = df_sales.sort('amount', ascending=False)
df_sales.show()

+--------+-----------+-------+-------+----------+------+-----------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|
+--------+-----------+-------+-------+----------+------+-----------------+
|    3486|       1646| Laptop|99999.0|2024-12-13| South|          89999.1|
|   69049|       6637| Laptop|99999.0|2025-05-10| South|          89999.1|
|    6874|       6345| Tablet|99997.0|2025-06-02| South|          89997.3|
|   96541|       9079| Laptop|99995.0|2024-05-26| North|          89995.5|
|   37866|        636| Mobile|99995.0|2025-05-07|  West|          89995.5|
|   96541|       9079| Laptop|99995.0|2024-05-26| North|          89995.5|
|    8104|       6810| Laptop|99993.0|2024-06-28|  West|          89993.7|
|   71825|       5731| Mobile|99993.0|2024-06-15| South|          89993.7|
|   78912|       9982| Laptop|99993.0|2025-12-31|  West|          89993.7|
|   20093|       6368|Desktop|99992.0|2025-06-13|  East|          89992.8|
|   70208|       8097| La

26. Sort the customer DataFrame by age in ascending order.

In [31]:
df_customer = df_customer.sort('age')
df_customer.show()

+-----------+-------------------+--------------------+---+--------------------+---------------------+
|customer_id|      customer_name|               email|age|       customer_city|customer_age_category|
+-----------+-------------------+--------------------+---+--------------------+---------------------+
|       1379|       James Kaiser|             unknown|  5|    East Christopher|                Young|
|       3300|    Marissa Stewart|johnhill@example.net|  5|     Lake Alexandria|                Young|
|       1402|   Jennifer Lindsey|urichardson@examp...|  5|        South Brooke|                Young|
|        210|      Bryan Johnson|carmenpeterson@ex...|  5|           Dianaland|                Young|
|       1561|    Jonathan Norman|christine54@examp...|  5|        Ramirezshire|                Young|
|        334|       Michael Frey|  ryan80@example.net|  5|            Loriport|                Young|
|       1657|       Melissa Hull|kleinchad@example...|  5|         West Nicole|   

27. Add a new dataset for customers and perform a union operation with the customer
DataFrame.

In [32]:
df_customer_duplicate = spark.read.csv("/content/customers_dirty.csv", header=True, inferSchema=True)
df_customer2 = spark.read.csv("/content/customer.txt", header=True, inferSchema=True)


union_data = df_customer_duplicate.union(df_customer2)
union_data.show()

+-----------+------------------+--------------------+---+-----------------+
|customer_id|     customer_name|               email|age|             city|
+-----------+------------------+--------------------+---+-----------------+
|        100| Jasmine Contreras| xmacias@example.org| 49|      Hollandtown|
|        101|     Michael Jones|michaelkhan@examp...| 27|  New Kennethstad|
|        102|  Dr. Jason Murray|penapatricia@exam...| 37|       West Megan|
|        103|   Karen Hernandez|                NULL| 28|   South Courtney|
|        104|  Kathleen Chapman|jessejones@exampl...| 45|      South Paige|
|        105|    Melissa Greene|deanaustin@exampl...| 24|      Charlesberg|
|        106|       Brenda Ruiz|                NULL| 32|        Lake Erin|
|        107| Dr. Paul Bautista|zbrennan@example.com| 37|      North Kayla|
|        108|      Samuel Garza|edward54@example.net| 33|     East Michael|
|        109|       Lori Morris|pricesusan@exampl...| 46|    New Jamesview|
|        110

28. Combine the sales DataFrame with another DataFrame containing additional sales
records.

In [33]:
df_sales_duplicate = spark.read.csv("/content/sales_dirty.csv", header=True, inferSchema=True)
df_sales2 = spark.read.csv("/content/sales.txt", header=True, inferSchema=True)
union_data2 = df_sales_duplicate.union(df_sales2)
union_data2.show()

+--------+-----------+-------+-------+----------+------+
|sales_id|customer_id|product| amount| sale_date|region|
+--------+-----------+-------+-------+----------+------+
|       1|       4296| Laptop|10299.0|2025-04-08| South|
|       2|       6307| Laptop|16842.0|2025-11-25|  East|
|       3|       5369|Desktop|64411.0|2025-04-13|  East|
|       4|       6125| Mobile|55543.0|2024-12-15| North|
|       5|       9800| Laptop|48545.0|2024-05-11|  East|
|       6|       5535| Laptop|73010.0|2025-10-06| North|
|       7|       8351| Laptop|15313.0|2024-07-13| North|
|       8|        247| Laptop|50976.0|2025-05-20| South|
|       9|       4285| Laptop|98067.0|2024-06-01|  West|
|      10|       2987| Laptop|11911.0|2025-04-08| North|
|      11|       2032|Desktop|79916.0|2025-05-05|  West|
|      12|       8810| Tablet|32820.0|2024-08-25| South|
|      13|       8932| Laptop|57096.0|2025-09-23| South|
|      14|       7953| Tablet|60109.0|2024-05-31|  West|
|      15|       5838|Desktop|3

29. Rank the sales records based on the amount column.

In [34]:
stat9 = df_sales.withColumn('rank', rank().over(Window.orderBy(desc('amount'))))
stat9.show()

+--------+-----------+-------+-------+----------+------+-----------------+----+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|rank|
+--------+-----------+-------+-------+----------+------+-----------------+----+
|    3486|       1646| Laptop|99999.0|2024-12-13| South|          89999.1|   1|
|   69049|       6637| Laptop|99999.0|2025-05-10| South|          89999.1|   1|
|    6874|       6345| Tablet|99997.0|2025-06-02| South|          89997.3|   3|
|   37866|        636| Mobile|99995.0|2025-05-07|  West|          89995.5|   4|
|   96541|       9079| Laptop|99995.0|2024-05-26| North|          89995.5|   4|
|   96541|       9079| Laptop|99995.0|2024-05-26| North|          89995.5|   4|
|    8104|       6810| Laptop|99993.0|2024-06-28|  West|          89993.7|   7|
|   71825|       5731| Mobile|99993.0|2024-06-15| South|          89993.7|   7|
|   78912|       9982| Laptop|99993.0|2025-12-31|  West|          89993.7|   7|
|   20093|       6368|Desktop|99992.0|20

30. Add a cumulative sum of amount for each product in the sales DataFrame.

In [35]:
windowframe = Window.partitionBy('product').orderBy('sale_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
stat10 = df_sales.withColumn('cumulative_sum', sum('amount').over(windowframe))
stat10.show()

+--------+-----------+-------+-------+----------+------+-----------------+--------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|cumulative_sum|
+--------+-----------+-------+-------+----------+------+-----------------+--------------+
|   82104|       3545| Laptop|99435.0|2024-03-25|  West|          89491.5|       99435.0|
|    1082|       6209| Laptop|97638.0|2024-03-25|  West|          87874.2|      197073.0|
|   18246|       1279| Laptop|97452.0|2024-03-25|  East|          87706.8|      294525.0|
|   99906|       8961| Laptop|96774.0|2024-03-25| South|          87096.6|      391299.0|
|   99999|       3502| Laptop|96751.0|2024-03-25| North|87075.90000000001|      488050.0|
|   98578|       3315| Laptop|96322.0|2024-03-25|  East|          86689.8|      584372.0|
|   92445|       3637| Laptop|95637.0|2024-03-25| North|          86073.3|      680009.0|
|    4646|       7397| Laptop|95402.0|2024-03-25|  East|          85861.8|      775411.0|
|    4646|

31. Add a column that calculates the difference between each customer's amount and the
average amount within their product group.

In [36]:
stat11 = df_sales.withColumn('avg_amount', avg('amount').over(Window.partitionBy('product')))
stat11 = stat11.withColumn('difference', col('amount') - col('avg_amount'))
stat11.show()

+--------+-----------+-------+-------+----------+------+-----------------+-----------------+-----------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|       avg_amount|       difference|
+--------+-----------+-------+-------+----------+------+-----------------+-----------------+-----------------+
|    3486|       1646| Laptop|99999.0|2024-12-13| South|          89999.1|48938.30477269097|51060.69522730903|
|   69049|       6637| Laptop|99999.0|2025-05-10| South|          89999.1|48938.30477269097|51060.69522730903|
|   96541|       9079| Laptop|99995.0|2024-05-26| North|          89995.5|48938.30477269097|51056.69522730903|
|   96541|       9079| Laptop|99995.0|2024-05-26| North|          89995.5|48938.30477269097|51056.69522730903|
|    8104|       6810| Laptop|99993.0|2024-06-28|  West|          89993.7|48938.30477269097|51054.69522730903|
|   78912|       9982| Laptop|99993.0|2025-12-31|  West|          89993.7|48938.30477269097|51054.69522730903|
|

32. Write the sales DataFrame to a partitioned Parquet file by region.

In [38]:
df_sales.write.mode('overwrite').partitionBy('region').parquet('/content/sales')

33. Partition the customer DataFrame by customer_city and save it as a CSV file.

In [39]:
df_customer.write.partitionBy('customer_city').csv('/content/customer')


34. Calculate the percentage contribution of each product to the total sales.

In [40]:
total_sales_count = df_sales.count()
stat12 = df_sales.groupBy('product').agg(count('product').alias('count'))
stat12 = stat12.withColumn('percentage_contribution', (col('count') / total_sales_count) * 100)
stat12.show()

+-------+-----+-----------------------+
|product|count|percentage_contribution|
+-------+-----+-----------------------+
| Laptop|68871|                  62.61|
| Mobile|13610|     12.372727272727273|
| Tablet|13854|     12.594545454545456|
|Desktop|13665|     12.422727272727272|
+-------+-----+-----------------------+



35. Extract the year from sale_date and group by year to calculate total sales.

In [41]:
stat13 = df_sales.withColumn('year', year('sale_date'))
stat13 = stat13.groupBy('year').agg(sum('amount'))
stat13.show()

+----+-------------+
|year|  sum(amount)|
+----+-------------+
|2025|2.685057983E9|
|2026| 6.16075015E8|
|2024|2.085611615E9|
+----+-------------+



36. Identify the most purchased product in each region.

In [43]:
stat14 = df_sales.groupBy('region', 'product').agg(count('sales_id').alias('count'))
stat14 = stat14.withColumn('rank', rank().over(Window.partitionBy('region').orderBy(desc('count'))))
stat14 = stat14.filter(stat14.rank == 1)
stat14.show()

+------+-------+-----+----+
|region|product|count|rank|
+------+-------+-----+----+
|  East| Laptop|17094|   1|
| North| Laptop|16955|   1|
| South| Laptop|17549|   1|
|  West| Laptop|17273|   1|
+------+-------+-----+----+



37. Add a column to show the difference between the highest and lowest sales for each
product.

In [44]:
stat15 = df_sales.groupBy('product').agg(max('amount').alias('max_amount'), min('amount').alias('min_amount'))
stat15 = stat15.withColumn('difference', col('max_amount') - col('min_amount'))
stat15.show()

+-------+----------+----------+----------+
|product|max_amount|min_amount|difference|
+-------+----------+----------+----------+
| Laptop|   99999.0|    -100.0|  100099.0|
| Mobile|   99995.0|    -100.0|  100095.0|
| Tablet|   99997.0|    -100.0|  100097.0|
|Desktop|   99992.0|    -100.0|  100092.0|
+-------+----------+----------+----------+



38. Write the result of the join between sales and customer to parquet file.

In [45]:
joined1.write.parquet('/content/joined1')

39. Identify products that were sold in the last 6 months.

In [46]:
max_date = df_sales.agg(max(col('sale_date'))).collect()
max_date = max_date[0][0]

In [48]:
max_date_past6 = date_sub(lit(max_date), 180)

In [49]:
products_last_6_months = df_sales.filter(col('sale_date') >= max_date_past6).select('product').distinct()
products_last_6_months.show()

+-------+
|product|
+-------+
| Laptop|
| Mobile|
| Tablet|
|Desktop|
+-------+



40. Calculate the average sales amount per customer.

In [50]:
stat16 = df_sales.groupBy('customer_id').agg(avg('amount')).orderBy('customer_id')
stat16.show()

+-----------+------------------+
|customer_id|       avg(amount)|
+-----------+------------------+
|        100|           52556.3|
|        101| 47497.11111111111|
|        102|43912.818181818184|
|        103|           56790.0|
|        104|          50940.25|
|        105|35891.444444444445|
|        106|         51254.625|
|        107|56999.818181818184|
|        108|42392.142857142855|
|        109|          51713.75|
|        110|34706.066666666666|
|        111|           54903.7|
|        112|           64333.3|
|        113|54109.555555555555|
|        114| 55503.53333333333|
|        115| 40038.11111111111|
|        116|63325.181818181816|
|        117|           49234.3|
|        118|38285.769230769234|
|        119| 43834.63636363636|
+-----------+------------------+
only showing top 20 rows


In [51]:
df_sales.createOrReplaceTempView('sales_table')
stat16_sql = spark.sql("SELECT customer_id, AVG(amount) as avg_amount FROM sales_table GROUP BY customer_id ORDER BY customer_id")
stat16_sql.show()

+-----------+------------------+
|customer_id|        avg_amount|
+-----------+------------------+
|        100|           52556.3|
|        101| 47497.11111111111|
|        102|43912.818181818184|
|        103|           56790.0|
|        104|          50940.25|
|        105|35891.444444444445|
|        106|         51254.625|
|        107|56999.818181818184|
|        108|42392.142857142855|
|        109|          51713.75|
|        110|34706.066666666666|
|        111|           54903.7|
|        112|           64333.3|
|        113|54109.555555555555|
|        114| 55503.53333333333|
|        115| 40038.11111111111|
|        116|63325.181818181816|
|        117|           49234.3|
|        118|38285.769230769234|
|        119| 43834.63636363636|
+-----------+------------------+
only showing top 20 rows
